In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-")
PHASE2_DIR = ROOT / "data_labeling" / "data_labels" / "GOES_2010_2018_NOAA_2019_2026"
NOAA_CL_DIR = ROOT / "data_labeling" / "data_labels" / "noaa_2010_2026" / "CL_Labels"
OUT_DIR = PHASE2_DIR / "CL_Labels_noaa_dates"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = PHASE2_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv"

def parse_timestamp(series: pd.Series) -> pd.Series:
    ts_str = series.str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
    return pd.to_datetime(ts_str, format="%Y.%m.%d_%H.%M.%S")

master_df = pd.read_csv(MASTER_CSV)
master_df["timestamp"] = parse_timestamp(master_df["label"])
master_df = master_df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded Phase 2 master CSV: {len(master_df):,} rows")
print(f"Range: {master_df['timestamp'].min()} -> {master_df['timestamp'].max()}")
print(f"FL={int((master_df['goes_class'] == 1).sum()):,} ({master_df['goes_class'].mean()*100:.2f}%)")

Loaded Phase 2 master CSV: 128,116 rows
Range: 2010-12-06 07:00:00 -> 2026-07-31 23:00:00
FL=26,999 (21.07%)


In [4]:
STAGE1_FILES = ["Stage1_train.csv", "Stage1_holdout.csv"]

splits = {}
summary_rows = []

for fname in STAGE1_FILES:
    ref_df = pd.read_csv(NOAA_CL_DIR / fname)
    ref_ts = parse_timestamp(ref_df["label"])
    t_min, t_max = ref_ts.min(), ref_ts.max()

    sub = master_df[(master_df["timestamp"] >= t_min) & (master_df["timestamp"] <= t_max)].copy()
    splits[fname] = sub

    fl = int((sub["goes_class"] == 1).sum())
    nf = int((sub["goes_class"] == 0).sum())
    ref_fl = int((ref_df["goes_class"] == 1).sum())

    summary_rows.append({
        "split": fname,
        "phase2_total": len(sub),
        "phase2_FL": fl,
        "phase2_NF": nf,
        "phase2_FL%": round(fl / len(sub) * 100, 2) if len(sub) else 0.0,
        "noaa_total": len(ref_df),
        "noaa_FL": ref_fl,
        "noaa_FL%": round(ref_fl / len(ref_df) * 100, 2),
        "start": t_min,
        "end": t_max,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,split,phase2_total,phase2_FL,phase2_NF,phase2_FL%,noaa_total,noaa_FL,noaa_FL%,start,end
0,Stage1_train.csv,46637,8645,37992,18.54,46638,8623,18.49,2010-12-06 07:00:00,2017-01-06 03:00:00
1,Stage1_holdout.csv,8230,330,7900,4.01,8230,330,4.01,2017-01-07 04:00:00,2017-12-31 23:00:00


In [5]:
s1_train = splits["Stage1_train.csv"]
s1_holdout = splits["Stage1_holdout.csv"]

overlap = set(s1_train["label"]) & set(s1_holdout["label"])
gap_hours = (s1_holdout["timestamp"].min() - s1_train["timestamp"].max()).total_seconds() / 3600.0

print(f"Overlap count: {len(overlap)}")
print(f"Gap Stage1_train -> Stage1_holdout: {gap_hours:.1f} hours")
assert len(overlap) == 0, "Overlap found!"
assert gap_hours >= 25.0, f"Expected >= 25h gap, got {gap_hours:.1f}h"
print("Stage 1 verification passed!")

Overlap count: 0
Gap Stage1_train -> Stage1_holdout: 25.0 hours
Stage 1 verification passed!


In [6]:
for fname, df_out in splits.items():
    out_path = OUT_DIR / fname
    df_out[["label", "goes_class"]].to_csv(out_path, index=False)
    print(f"Saved {out_path} -> {len(df_out):,} rows")

Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage1_train.csv -> 46,637 rows
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage1_holdout.csv -> 8,230 rows


stage 2,3 and test set

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-")
PHASE2_DIR = ROOT / "data_labeling" / "data_labels" / "GOES_2010_2018_NOAA_2019_2026"
OUT_DIR = PHASE2_DIR / "CL_Labels_noaa_dates"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOC_CSV = PHASE2_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_with_loc.csv"
BIN_CSV = PHASE2_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv"

def parse_timestamp(series: pd.Series) -> pd.Series:
    ts_str = series.str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
    return pd.to_datetime(ts_str, format="%Y.%m.%d_%H.%M.%S")

def goes_to_bucket_and_binary(goes_str: str):
    gc = str(goes_str).strip().upper()
    if gc == "NF":
        return "NF", 0
    letter, mag = gc[0], float(gc[1:])
    if letter in {"A", "B", "C"}:
        return letter, 0
    if letter == "M":
        # M0.9 (< M1.0) maps to C-class / binary 0
        return ("M", 1) if mag >= 1.0 else ("C", 0)
    if letter == "X":
        return "X", 1
    raise ValueError(f"Unknown GOES class: {goes_str!r}")

master_df = pd.read_csv(LOC_CSV)
master_df["timestamp"] = parse_timestamp(master_df["label"])
parsed = master_df["goes_class"].apply(goes_to_bucket_and_binary)
master_df["goes_bucket"] = [b for b, _ in parsed]
master_df["binary_label"] = [y for _, y in parsed]

# Cross-check against complete_binary.csv
bin_df = pd.read_csv(BIN_CSV)
merged = master_df[["label", "binary_label"]].merge(bin_df, on="label", how="inner")
assert len(merged) == len(master_df) == len(bin_df), "Row count mismatch!"
assert (merged["binary_label"] == merged["goes_class"]).all(), "Label mismatch between with_loc and binary CSVs!"

master_df = master_df.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded & verified {len(master_df):,} images from complete_with_loc.csv (0 mismatches).")

Loaded & verified 128,116 images from complete_with_loc.csv (0 mismatches).


Define Split Boundaries & Slice Stage 2, Stage 3, and 8-Month Test

In [2]:
# Keep existing Stage 1 splits untouched so they match Stage 1 model 141626 exactly
s1_train_labels = set(pd.read_csv(OUT_DIR / "Stage1_train.csv")["label"])
s1_hold_labels = set(pd.read_csv(OUT_DIR / "Stage1_holdout.csv")["label"])

SPLIT_SPECS = [
    ("Stage1_train.csv",     lambda df: df[df["label"].isin(s1_train_labels)]),
    ("Stage1_holdout.csv",   lambda df: df[df["label"].isin(s1_hold_labels)]),
    ("Stage2_train.csv",     lambda df: df[(df["timestamp"] >= "2018-01-02 00:00:00") & (df["timestamp"] <= "2023-03-08 23:59:59")]),
    ("Stage2_holdout.csv",   lambda df: df[(df["timestamp"] >= "2023-03-10 00:00:00") & (df["timestamp"] <= "2023-06-30 23:59:59")]),
    ("Stage3_train.csv",     lambda df: df[(df["timestamp"] >= "2023-07-02 00:00:00") & (df["timestamp"] <= "2025-06-30 23:59:59")]),
    ("Stage3_holdout.csv",   lambda df: df[(df["timestamp"] >= "2025-07-02 00:00:00") & (df["timestamp"] <= "2025-11-30 23:59:59")]),
    ("test_2025_2026_8m.csv",lambda df: df[(df["timestamp"] >= "2025-12-02 00:00:00") & (df["timestamp"] <= "2026-07-31 23:59:59")]),
]

splits = {name: fn(master_df).copy().sort_values("timestamp").reset_index(drop=True) for name, fn in SPLIT_SPECS}
print("Created in-memory splits:", {k: len(v) for k, v in splits.items()})

Created in-memory splits: {'Stage1_train.csv': 46637, 'Stage1_holdout.csv': 8230, 'Stage2_train.csv': 44126, 'Stage2_holdout.csv': 2675, 'Stage3_train.csv': 17141, 'Stage3_holdout.csv': 3607, 'test_2025_2026_8m.csv': 5556}


Verify Zero Overlap, 
≥
25
h
≥25h Gaps

In [3]:
seen_labels = set()
prev_end = None
summary = []

for name, df_split in splits.items():
    labels = set(df_split["label"])
    overlap = len(seen_labels & labels)
    assert overlap == 0, f"Overlap detected in {name}: {overlap}"
    seen_labels |= labels

    t_start, t_end = df_split["timestamp"].min(), df_split["timestamp"].max()
    gap_h = (t_start - prev_end).total_seconds() / 3600.0 if prev_end is not None else None
    if gap_h is not None:
        assert gap_h >= 25.0, f"Gap before {name} is {gap_h:.1f}h (< 25h)!"
    prev_end = t_end

    vc = df_split["goes_bucket"].value_counts().to_dict()
    total = len(df_split)
    fl = int((df_split["binary_label"] == 1).sum())
    non_fl = total - fl

    summary.append({
        "Split": name,
        "Start": str(t_start)[:10],
        "End": str(t_end)[:10],
        "Gap_Before_h": f"{gap_h:.1f}" if gap_h is not None else "-",
        "Total": total,
        "NF": vc.get("NF", 0),
        "A": vc.get("A", 0),
        "B": vc.get("B", 0),
        "C": vc.get("C", 0),
        "NonFlare(0)": non_fl,
        "M": vc.get("M", 0),
        "X": vc.get("X", 0),
        "Flare(1)": fl,
        "FL_%": round(100.0 * fl / total, 1),
        "NF:FL": round(non_fl / fl, 1) if fl else None,
    })

summary_df = pd.DataFrame(summary)
print("ALL OVERLAP & >=25h GAP ASSERTIONS PASSED!")
display(summary_df)

ALL OVERLAP & >=25h GAP ASSERTIONS PASSED!


,Split,Start,End,Gap_Before_h,Total,NF,A,B,C,NonFlare(0),M,X,Flare(1),FL_%,NF:FL
0,Stage1_train.csv,2010-12-06,2017-01-06,-,46637,3223,0,11186,23583,37992,7838,807,8645,18.5,4.4
1,Stage1_holdout.csv,2017-01-07,2017-12-31,25.0,8230,3364,0,3311,1225,7900,273,57,330,4.0,23.9
2,Stage2_train.csv,2018-01-02,2023-03-08,25.0,44126,19547,2193,9093,9512,40345,3426,355,3781,8.6,10.7
3,Stage2_holdout.csv,2023-03-10,2023-06-30,25.0,2675,5,0,18,1474,1497,1130,48,1178,44.0,1.3
4,Stage3_train.csv,2023-07-02,2025-06-30,25.0,17141,61,0,97,7110,7268,8602,1271,9873,57.6,0.7
5,Stage3_holdout.csv,2025-07-02,2025-11-30,25.0,3607,20,0,17,2188,2225,1237,145,1382,38.3,1.6
6,test_2025_2026_8m.csv,2025-12-02,2026-07-31,25.0,5556,18,0,193,3562,3773,1519,264,1783,32.1,2.1


In [4]:
FILES_TO_WRITE = [
    "Stage2_train.csv",
    "Stage2_holdout.csv",
    "Stage3_train.csv",
    "Stage3_holdout.csv",
    "test_2025_2026_8m.csv",
]

for fname in FILES_TO_WRITE:
    out_df = splits[fname][["label", "binary_label"]].rename(columns={"binary_label": "goes_class"})
    out_path = OUT_DIR / fname
    out_df.to_csv(out_path, index=False)
    print(f"Saved {out_path} -> {len(out_df):,} rows (FL={int((out_df['goes_class'] == 1).sum()):,})")

Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage2_train.csv -> 44,126 rows (FL=3,781)
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage2_holdout.csv -> 2,675 rows (FL=1,178)
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage3_train.csv -> 17,141 rows (FL=9,873)
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage3_holdout.csv -> 3,607 rows (FL=1,382)
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_da